In [1]:
import sys
import gc
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../src")

import models
import replay_buffer
import train
import evaluation

importlib.reload(models)
importlib.reload(replay_buffer)
importlib.reload(train)
importlib.reload(evaluation)

from models import DQN
from train import ConfigDQN, entrenar_dqn
from evaluation import evaluar_modelo

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Dispositivo: {device}")

Dispositivo: mps


# Ajuste de hiperparámetros de DQN + PER + n-step

En este notebook se analiza el efecto de dos hiperparámetros:

- El grado de priorización del replay buffer, controlado por `per_alpha`.
- El número de pasos utilizados para acumular recompensas, controlado por `n_step`.

Los demás hiperparámetros se mantendrán constantes para realizar una comparación controlada.

In [2]:
import gc

# Liberar modelos grandes que ya no necesitamos en memoria
objetos_temporales = [
    "resultado_dqn_per_3step",
    "mejor_dqn_per_3step",
    "checkpoint_3step",
]

for nombre in objetos_temporales:
    globals().pop(nombre, None)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_per_alpha04_3step = ConfigDQN(
    nombre_experimento="v7_dqn_per_a04_3step",
    total_pasos=1_000_000,

    seed=42,
    gamma=0.99,
    n_step=3,
    learning_rate=1e-4,
    batch_size=32,
    frecuencia_entrenamiento=4,

    capacidad_buffer=20_000,
    inicio_entrenamiento=10_000,
    frecuencia_actualizacion_target=10_000,
    gradient_clip=10.0,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=250_000,

    usar_per=True,
    per_alpha=0.4,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=500_000,
    per_epsilon=1e-6,

    frecuencia_evaluacion=50_000,
    episodios_evaluacion=5,
    seed_evaluacion=1_000,
    frecuencia_log=1_000,

    terminal_on_life_loss=True,
    clip_reward=True,
)

resultado_per_alpha04_3step = entrenar_dqn(
    config=config_per_alpha04_3step,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
)

print(
    "\nENTRENAMIENTO V7 FINALIZADO"
)

print(
    "Mejor promedio de evaluación: "
    f"{resultado_per_alpha04_3step['mejor_promedio_evaluacion']:.2f}"
)

A.L.E: Arcade Learning Environment (version 0.10.1+6a7e0ae)
[Powered by Stella]


Replay buffer: priorizado | alpha=0.4 | beta inicial=0.4
Retorno utilizado: 3-step
Paso 10,000/1,000,000 | episodio=49 | epsilon=0.964 | loss=0.0550 | Q=0.006
Paso 11,000/1,000,000 | episodio=53 | epsilon=0.960 | loss=0.0331 | Q=0.128
Paso 12,000/1,000,000 | episodio=59 | epsilon=0.957 | loss=0.0270 | Q=0.138
Paso 13,000/1,000,000 | episodio=62 | epsilon=0.953 | loss=0.0308 | Q=0.126
Paso 14,000/1,000,000 | episodio=67 | epsilon=0.950 | loss=0.0261 | Q=0.158
Paso 15,000/1,000,000 | episodio=71 | epsilon=0.946 | loss=0.0160 | Q=0.170
Paso 16,000/1,000,000 | episodio=77 | epsilon=0.942 | loss=0.0417 | Q=0.188
Paso 17,000/1,000,000 | episodio=81 | epsilon=0.939 | loss=0.0172 | Q=0.149
Paso 18,000/1,000,000 | episodio=84 | epsilon=0.935 | loss=0.0445 | Q=0.160
Paso 19,000/1,000,000 | episodio=88 | epsilon=0.932 | loss=0.0268 | Q=0.192
Paso 20,000/1,000,000 | episodio=92 | epsilon=0.928 | loss=0.0418 | Q=0.218
Paso 21,000/1,000,000 | episodio=97 | epsilon=0.924 | loss=0.0049 | Q=0.253
Paso 

In [3]:
ruta_evaluaciones_v7 = Path(
    "../logs/entrenamientos/"
    "v7_dqn_per_a04_3step/evaluaciones.csv"
)

df_evaluaciones_v7 = pd.read_csv(
    ruta_evaluaciones_v7
)

df_evaluaciones_v7 = (
    df_evaluaciones_v7
    .sort_values("paso_global")
    .reset_index(drop=True)
)

display(df_evaluaciones_v7)

mejor_evaluacion_v7 = df_evaluaciones_v7.loc[
    df_evaluaciones_v7["promedio"].idxmax()
]

ultimas_tres_v7 = (
    df_evaluaciones_v7
    .tail(3)["promedio"]
    .mean()
)

print("\nRESUMEN DE V7")
print(
    f"Mejor paso: "
    f"{int(mejor_evaluacion_v7['paso_global']):,}"
)
print(
    f"Mejor promedio: "
    f"{mejor_evaluacion_v7['promedio']:.2f}"
)
print(
    f"Mediana del mejor checkpoint: "
    f"{mejor_evaluacion_v7['mediana']:.2f}"
)
print(
    f"Desviación: "
    f"{mejor_evaluacion_v7['desviacion']:.2f}"
)
print(
    f"Mínimo: "
    f"{mejor_evaluacion_v7['minimo']:.2f}"
)
print(
    f"Máximo: "
    f"{mejor_evaluacion_v7['maximo']:.2f}"
)
print(
    f"Promedio de las últimas tres evaluaciones: "
    f"{ultimas_tres_v7:.2f}"
)

ruta_mejor_v7 = Path(
    "../models/v7_dqn_per_a04_3step/"
    "mejor_modelo.pt"
)

print(
    "\nCheckpoint disponible: "
    f"{'OK' if ruta_mejor_v7.exists() else 'NO ENCONTRADO'}"
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,50000,149.0,135.0,52.478567,95.0,240.0
1,100000,191.0,160.0,116.206712,70.0,330.0
2,150000,297.0,295.0,116.730459,135.0,460.0
3,200000,383.0,305.0,123.393679,255.0,545.0
4,250000,302.0,305.0,80.659779,175.0,420.0
5,300000,242.0,160.0,113.604577,155.0,445.0
6,350000,250.0,200.0,85.029407,160.0,365.0
7,400000,478.0,465.0,124.923977,325.0,630.0
8,450000,267.0,220.0,113.428392,130.0,425.0
9,500000,294.0,330.0,103.121288,160.0,415.0



RESUMEN DE V7
Mejor paso: 1,000,000
Mejor promedio: 493.00
Mediana del mejor checkpoint: 480.00
Desviación: 114.04
Mínimo: 365.00
Máximo: 705.00
Promedio de las últimas tres evaluaciones: 438.67

Checkpoint disponible: OK


## Extensión de V7 hasta 1,500,000 pasos

Debido a que el mejor resultado de V7 ocurrió en el último checkpoint y las evaluaciones finales mostraron una tendencia favorable, se continúa el entrenamiento durante 500,000 pasos adicionales.

El replay buffer comienza vacío al reanudar, por lo que se recopilan 10,000 experiencias nuevas antes de retomar las actualizaciones.

In [4]:
import importlib
import gc

import train
importlib.reload(train)

from train import ConfigDQN, entrenar_dqn

ruta_checkpoint_v7 = Path(
    "../models/v7_dqn_per_a04_3step/"
    "checkpoint_final.pt"
)

if not ruta_checkpoint_v7.exists():
    raise FileNotFoundError(
        f"No se encontró el checkpoint: "
        f"{ruta_checkpoint_v7}"
    )

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_v7_extendido = ConfigDQN(
    nombre_experimento=(
        "v7_dqn_per_a04_3step_extendido"
    ),
    total_pasos=1_500_000,

    seed=42,
    gamma=0.99,
    n_step=3,
    learning_rate=1e-4,
    batch_size=32,
    frecuencia_entrenamiento=4,

    capacidad_buffer=20_000,
    inicio_entrenamiento=10_000,
    frecuencia_actualizacion_target=10_000,
    gradient_clip=10.0,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=250_000,

    usar_per=True,
    per_alpha=0.4,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=500_000,
    per_epsilon=1e-6,

    frecuencia_evaluacion=50_000,
    episodios_evaluacion=5,
    seed_evaluacion=1_000,
    frecuencia_log=1_000,

    terminal_on_life_loss=True,
    clip_reward=True,
)

resultado_v7_extendido = entrenar_dqn(
    config=config_v7_extendido,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
    ruta_checkpoint_inicial=(
        ruta_checkpoint_v7
    ),
)

print(
    "\nEXTENSIÓN DE V7 FINALIZADA"
)

print(
    "Mejor promedio durante la extensión: "
    f"{resultado_v7_extendido['mejor_promedio_evaluacion']:.2f}"
)

Reanudando entrenamiento desde el paso 1,000,000 (checkpoint: ../models/v7_dqn_per_a04_3step/checkpoint_final.pt)
Replay buffer: priorizado | alpha=0.4 | beta inicial=0.4
Retorno utilizado: 3-step
Paso 1,010,000/1,500,000 | episodio=25 | epsilon=0.100 | loss=0.0399 | Q=5.046
Paso 1,011,000/1,500,000 | episodio=28 | epsilon=0.100 | loss=0.0203 | Q=4.656
Paso 1,012,000/1,500,000 | episodio=31 | epsilon=0.100 | loss=0.0062 | Q=5.263
Paso 1,013,000/1,500,000 | episodio=33 | epsilon=0.100 | loss=0.0217 | Q=5.342
Paso 1,014,000/1,500,000 | episodio=37 | epsilon=0.100 | loss=0.0265 | Q=5.254
Paso 1,015,000/1,500,000 | episodio=39 | epsilon=0.100 | loss=0.0082 | Q=5.395
Paso 1,016,000/1,500,000 | episodio=40 | epsilon=0.100 | loss=0.0164 | Q=4.702
Paso 1,017,000/1,500,000 | episodio=43 | epsilon=0.100 | loss=0.0022 | Q=5.224
Paso 1,018,000/1,500,000 | episodio=47 | epsilon=0.100 | loss=0.0056 | Q=4.788
Paso 1,019,000/1,500,000 | episodio=50 | epsilon=0.100 | loss=0.0056 | Q=4.933
Paso 1,020,00

In [5]:
ruta_evaluaciones_v7_ext = Path(
    "../logs/entrenamientos/"
    "v7_dqn_per_a04_3step_extendido/"
    "evaluaciones.csv"
)

df_evaluaciones_v7_ext = pd.read_csv(
    ruta_evaluaciones_v7_ext
)

df_evaluaciones_v7_ext = (
    df_evaluaciones_v7_ext
    .sort_values("paso_global")
    .reset_index(drop=True)
)

display(df_evaluaciones_v7_ext)

mejor_evaluacion_v7_ext = (
    df_evaluaciones_v7_ext.loc[
        df_evaluaciones_v7_ext[
            "promedio"
        ].idxmax()
    ]
)

print("\nMEJOR EVALUACIÓN DE V7 EXTENDIDO")
print(
    f"Paso: "
    f"{int(mejor_evaluacion_v7_ext['paso_global']):,}"
)
print(
    f"Promedio: "
    f"{mejor_evaluacion_v7_ext['promedio']:.2f}"
)
print(
    f"Mediana: "
    f"{mejor_evaluacion_v7_ext['mediana']:.2f}"
)
print(
    f"Desviación: "
    f"{mejor_evaluacion_v7_ext['desviacion']:.2f}"
)
print(
    f"Mínimo: "
    f"{mejor_evaluacion_v7_ext['minimo']:.2f}"
)
print(
    f"Máximo: "
    f"{mejor_evaluacion_v7_ext['maximo']:.2f}"
)

ruta_mejor_v7_ext = Path(
    "../models/"
    "v7_dqn_per_a04_3step_extendido/"
    "mejor_modelo.pt"
)

print(
    "\nCheckpoint disponible: "
    f"{'OK' if ruta_mejor_v7_ext.exists() else 'NO ENCONTRADO'}"
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,1050000,455.0,450.0,122.759928,315.0,615.0
1,1100000,508.0,425.0,254.727305,285.0,990.0
2,1150000,475.0,475.0,60.580525,385.0,575.0
3,1200000,447.0,380.0,150.086642,345.0,745.0
4,1250000,462.0,470.0,92.336342,290.0,545.0
5,1300000,485.0,455.0,96.332757,350.0,600.0
6,1350000,427.0,325.0,197.651208,225.0,775.0
7,1400000,631.0,550.0,279.291962,340.0,1100.0
8,1450000,565.0,580.0,205.377701,330.0,885.0
9,1500000,465.0,375.0,149.432259,310.0,720.0



MEJOR EVALUACIÓN DE V7 EXTENDIDO
Paso: 1,400,000
Promedio: 631.00
Mediana: 550.00
Desviación: 279.29
Mínimo: 340.00
Máximo: 1100.00

Checkpoint disponible: OK


In [6]:
import evaluation
importlib.reload(evaluation)

from evaluation import evaluar_modelo
from models import DQN

N_EPISODIOS_EVALUACION = 30
SEMILLA_BASE_EVALUACION = 42

checkpoint_v7_ext = torch.load(
    ruta_mejor_v7_ext,
    map_location=device,
    weights_only=True,
)

mejor_v7_extendido = DQN(
    n_acciones=6
).to(device)

mejor_v7_extendido.load_state_dict(
    checkpoint_v7_ext[
        "modelo_online_state_dict"
    ]
)

mejor_v7_extendido.eval()

print(
    f"Checkpoint cargado desde el paso: "
    f"{checkpoint_v7_ext['paso']:,}"
)

print(
    "\nEvaluando V7 extendido durante "
    f"{N_EPISODIOS_EVALUACION} episodios...\n"
)

(
    resultados_v7_extendido,
    resumen_v7_extendido,
) = evaluar_modelo(
    modelo=mejor_v7_extendido,
    config=config_v7_extendido,
    device=device,
    n_episodios=N_EPISODIOS_EVALUACION,
    seed_base=SEMILLA_BASE_EVALUACION,
)

df_v7_extendido = pd.DataFrame(
    resultados_v7_extendido
)

df_v7_extendido["agente"] = (
    "DQN + PER a=0.4 + 3-step extendido"
)

df_v7_extendido = df_v7_extendido[
    [
        "agente",
        "episodio",
        "seed",
        "recompensa_total",
        "pasos",
        "terminated",
        "truncated",
    ]
]

print("RESUMEN DE 30 EPISODIOS")

for metrica, valor in resumen_v7_extendido.items():
    print(f"{metrica}: {valor:.2f}")

display(df_v7_extendido.head(10))

Checkpoint cargado desde el paso: 1,400,000

Evaluando V7 extendido durante 30 episodios...

RESUMEN DE 30 EPISODIOS
promedio: 421.83
mediana: 430.00
desviacion: 144.56
minimo: 180.00
maximo: 960.00


,agente,episodio,seed,recompensa_total,pasos,terminated,truncated
0,DQN + PER a=0.4 + 3-step extendido,0,42,630.0,1109,True,False
1,DQN + PER a=0.4 + 3-step extendido,1,43,400.0,726,True,False
2,DQN + PER a=0.4 + 3-step extendido,2,44,455.0,911,True,False
3,DQN + PER a=0.4 + 3-step extendido,3,45,280.0,645,True,False
4,DQN + PER a=0.4 + 3-step extendido,4,46,490.0,882,True,False
5,DQN + PER a=0.4 + 3-step extendido,5,47,330.0,613,True,False
6,DQN + PER a=0.4 + 3-step extendido,6,48,295.0,533,True,False
7,DQN + PER a=0.4 + 3-step extendido,7,49,485.0,880,True,False
8,DQN + PER a=0.4 + 3-step extendido,8,50,250.0,656,True,False
9,DQN + PER a=0.4 + 3-step extendido,9,51,960.0,1674,True,False


In [7]:
ruta_resultados_v7_ext = Path(
    "../logs/evaluacion_v7_extendido.csv"
)

df_v7_extendido.to_csv(
    ruta_resultados_v7_ext,
    index=False,
)

df_alpha06 = pd.read_csv(
    "../logs/evaluacion_dqn_per_3step.csv"
)

comparacion_alpha = (
    pd.concat(
        [
            df_alpha06,
            df_v7_extendido,
        ],
        ignore_index=True,
    )
    .pivot(
        index="seed",
        columns="agente",
        values="recompensa_total",
    )
    .dropna()
)

nombre_alpha06 = "DQN + PER + 3-step"
nombre_alpha04 = (
    "DQN + PER a=0.4 + 3-step extendido"
)

diferencia = (
    comparacion_alpha[nombre_alpha04]
    - comparacion_alpha[nombre_alpha06]
)

print(
    f"Victorias {nombre_alpha06}:",
    (diferencia < 0).sum(),
)

print(
    f"Victorias {nombre_alpha04}:",
    (diferencia > 0).sum(),
)

print(
    "Empates:",
    (diferencia == 0).sum(),
)

# Comparación del criterio mejor-de-cinco
df_competencia_alpha = pd.concat(
    [
        df_alpha06,
        df_v7_extendido,
    ],
    ignore_index=True,
)

df_competencia_alpha["bloque_de_5"] = (
    df_competencia_alpha["episodio"] // 5
) + 1

mejores_de_5_alpha = (
    df_competencia_alpha
    .groupby(
        ["agente", "bloque_de_5"],
        as_index=False,
    )
    .agg(
        mejor_recompensa=(
            "recompensa_total",
            "max",
        ),
        promedio_bloque=(
            "recompensa_total",
            "mean",
        ),
    )
)

resumen_competencia_alpha = (
    mejores_de_5_alpha
    .groupby("agente")
    .agg(
        mejor_de_5_promedio=(
            "mejor_recompensa",
            "mean",
        ),
        mejor_de_5_mediana=(
            "mejor_recompensa",
            "median",
        ),
        menor_mejor_de_5=(
            "mejor_recompensa",
            "min",
        ),
        mayor_mejor_de_5=(
            "mejor_recompensa",
            "max",
        ),
    )
    .round(2)
    .reset_index()
)

print(
    "\nCOMPARACIÓN DEL MEJOR DE CINCO"
)

display(resumen_competencia_alpha)

Victorias DQN + PER + 3-step: 17
Victorias DQN + PER a=0.4 + 3-step extendido: 12
Empates: 1

COMPARACIÓN DEL MEJOR DE CINCO


,agente,mejor_de_5_promedio,mejor_de_5_mediana,menor_mejor_de_5,mayor_mejor_de_5
0,DQN + PER + 3-step,611.67,582.5,490.0,770.0
1,DQN + PER a=0.4 + 3-step extendido,617.50,562.5,465.0,960.0


## Experimento V8: PER con alpha=0.6 y retorno 5-step

Esta iteración mantiene la configuración de priorización que produjo el mejor rendimiento general (`per_alpha=0.6`) y modifica únicamente el horizonte del retorno:

- V6: `n_step=3`
- V8: `n_step=5`

El objetivo es determinar si incorporar cinco recompensas consecutivas permite propagar mejor la señal de recompensa. Los demás hiperparámetros se mantienen constantes para realizar una comparación controlada.

In [8]:
import gc
import importlib

import train
importlib.reload(train)

from train import ConfigDQN, entrenar_dqn
from models import DQN

# Liberar modelos anteriores
objetos_temporales = [
    "resultado_v7_extendido",
    "mejor_v7_extendido",
    "checkpoint_v7_ext",
]

for nombre in objetos_temporales:
    globals().pop(nombre, None)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_per_alpha06_5step = ConfigDQN(
    nombre_experimento="v8_dqn_per_a06_5step",
    total_pasos=1_000_000,

    seed=42,
    gamma=0.99,
    n_step=5,
    learning_rate=1e-4,
    batch_size=32,
    frecuencia_entrenamiento=4,

    capacidad_buffer=20_000,
    inicio_entrenamiento=10_000,
    frecuencia_actualizacion_target=10_000,
    gradient_clip=10.0,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=250_000,

    usar_per=True,
    per_alpha=0.6,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=500_000,
    per_epsilon=1e-6,

    frecuencia_evaluacion=50_000,
    episodios_evaluacion=5,
    seed_evaluacion=1_000,
    frecuencia_log=1_000,

    terminal_on_life_loss=True,
    clip_reward=True,
)

resultado_per_alpha06_5step = entrenar_dqn(
    config=config_per_alpha06_5step,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
)

print(
    "\nENTRENAMIENTO V8 FINALIZADO"
)

print(
    "Mejor promedio de evaluación: "
    f"{resultado_per_alpha06_5step['mejor_promedio_evaluacion']:.2f}"
)

Replay buffer: priorizado | alpha=0.6 | beta inicial=0.4
Retorno utilizado: 5-step
Paso 10,000/1,000,000 | episodio=49 | epsilon=0.964 | loss=0.0857 | Q=-0.007
Paso 11,000/1,000,000 | episodio=52 | epsilon=0.960 | loss=0.0158 | Q=0.190
Paso 12,000/1,000,000 | episodio=57 | epsilon=0.957 | loss=0.0609 | Q=0.219
Paso 13,000/1,000,000 | episodio=63 | epsilon=0.953 | loss=0.0386 | Q=0.228
Paso 14,000/1,000,000 | episodio=66 | epsilon=0.950 | loss=0.0377 | Q=0.255
Paso 15,000/1,000,000 | episodio=72 | epsilon=0.946 | loss=0.0370 | Q=0.234
Paso 16,000/1,000,000 | episodio=77 | epsilon=0.942 | loss=0.0211 | Q=0.181
Paso 17,000/1,000,000 | episodio=79 | epsilon=0.939 | loss=0.0617 | Q=0.229
Paso 18,000/1,000,000 | episodio=84 | epsilon=0.935 | loss=0.0366 | Q=0.217
Paso 19,000/1,000,000 | episodio=92 | epsilon=0.932 | loss=0.0196 | Q=0.216
Paso 20,000/1,000,000 | episodio=98 | epsilon=0.928 | loss=0.0356 | Q=0.261
Paso 21,000/1,000,000 | episodio=103 | epsilon=0.924 | loss=0.0210 | Q=0.505
Pas

In [9]:
from pathlib import Path
import pandas as pd
from IPython.display import display

RUTA_LOG_V8 = Path(
    "../logs/entrenamientos/v8_dqn_per_a06_5step/evaluaciones.csv"
)
RUTA_MODELO_V8 = Path(
    "../models/v8_dqn_per_a06_5step/mejor_modelo.pt"
)

df_evaluaciones_v8 = pd.read_csv(RUTA_LOG_V8)

display(df_evaluaciones_v8)

mejor_fila_v8 = df_evaluaciones_v8.loc[
    df_evaluaciones_v8["promedio"].idxmax()
]

promedio_ultimas_tres_v8 = (
    df_evaluaciones_v8["promedio"]
    .tail(3)
    .mean()
)

print("\nMEJOR EVALUACIÓN DE V8")
print(f"Paso: {int(mejor_fila_v8['paso_global']):,}")
print(f"Promedio: {mejor_fila_v8['promedio']:.2f}")
print(f"Mediana: {mejor_fila_v8['mediana']:.2f}")
print(f"Desviación: {mejor_fila_v8['desviacion']:.2f}")
print(f"Mínimo: {mejor_fila_v8['minimo']:.2f}")
print(f"Máximo: {mejor_fila_v8['maximo']:.2f}")

print(
    "\nPromedio de las últimas tres evaluaciones: "
    f"{promedio_ultimas_tres_v8:.2f}"
)

print(
    "\nCheckpoint disponible:",
    "OK" if RUTA_MODELO_V8.exists() else "NO ENCONTRADO",
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,50000,168.0,110.0,111.292408,90.0,385.0
1,100000,192.0,155.0,77.304592,125.0,335.0
2,150000,354.0,390.0,130.820488,160.0,535.0
3,200000,250.0,155.0,151.129084,155.0,545.0
4,250000,453.0,360.0,218.965751,245.0,870.0
5,300000,326.0,295.0,112.178429,165.0,465.0
6,350000,242.0,290.0,106.423682,95.0,350.0
7,400000,524.0,520.0,235.868607,270.0,945.0
8,450000,440.0,455.0,98.640762,290.0,545.0
9,500000,470.0,515.0,125.777581,225.0,575.0



MEJOR EVALUACIÓN DE V8
Paso: 1,000,000
Promedio: 525.00
Mediana: 575.00
Desviación: 177.12
Mínimo: 260.00
Máximo: 775.00

Promedio de las últimas tres evaluaciones: 461.67

Checkpoint disponible: OK


In [13]:
import torch
from pathlib import Path

from models import DQN
from evaluation import evaluar_modelo


RUTA_CHECKPOINT_V8 = Path(
    "../models/v8_dqn_per_a06_5step/mejor_modelo.pt"
)

checkpoint_v8 = torch.load(
    RUTA_CHECKPOINT_V8,
    map_location=device,
    weights_only=True,
)

mejor_dqn_v8 = DQN(
    n_acciones=6
).to(device)

mejor_dqn_v8.load_state_dict(
    checkpoint_v8["modelo_online_state_dict"]
)

mejor_dqn_v8.eval()

print(
    "Checkpoint cargado desde el paso:",
    f"{checkpoint_v8['paso']:,}",
)

Checkpoint cargado desde el paso: 1,000,000


In [16]:
N_EPISODIOS_EVALUACION = 30
SEMILLA_BASE_EVALUACION = 42

print(
    f"\nEvaluando V8: DQN + PER + 5-step durante "
    f"{N_EPISODIOS_EVALUACION} episodios...\n"
)

resultados_v8, resumen_v8 = evaluar_modelo(
    modelo=mejor_dqn_v8,
    config=config_per_alpha06_5step,
    device=device,
    n_episodios=N_EPISODIOS_EVALUACION,
    seed_base=SEMILLA_BASE_EVALUACION,
)

df_v8 = pd.DataFrame(resultados_v8)

df_v8["agente"] = "DQN + PER + 5-step"

df_v8 = df_v8[
    [
        "agente",
        "episodio",
        "seed",
        "recompensa_total",
        "pasos",
        "terminated",
        "truncated",
    ]
]

print("RESUMEN DE 30 EPISODIOS")

for metrica, valor in resumen_v8.items():
    print(f"{metrica}: {valor:.2f}")

display(df_v8.head(10))


Evaluando V8: DQN + PER + 5-step durante 30 episodios...

RESUMEN DE 30 EPISODIOS
promedio: 440.17
mediana: 410.00
desviacion: 124.38
minimo: 255.00
maximo: 805.00


,agente,episodio,seed,recompensa_total,pasos,terminated,truncated
0,DQN + PER + 5-step,0,42,325.0,541,True,False
1,DQN + PER + 5-step,1,43,390.0,579,True,False
2,DQN + PER + 5-step,2,44,405.0,761,True,False
3,DQN + PER + 5-step,3,45,300.0,712,True,False
4,DQN + PER + 5-step,4,46,255.0,657,True,False
5,DQN + PER + 5-step,5,47,515.0,606,True,False
6,DQN + PER + 5-step,6,48,540.0,625,True,False
7,DQN + PER + 5-step,7,49,435.0,880,True,False
8,DQN + PER + 5-step,8,50,395.0,711,True,False
9,DQN + PER + 5-step,9,51,305.0,600,True,False


In [17]:
import gc
from dataclasses import replace
from pathlib import Path

import torch

from train import entrenar_dqn
from models import DQN

config_v8_extendido = replace(
    config_per_alpha06_5step,
    nombre_experimento="v8_dqn_per_a06_5step_extendido",
    total_pasos=2_000_000,
)

directorio_v8_original = Path(
    "../models/v8_dqn_per_a06_5step"
)

ruta_checkpoint_v8 = (
    directorio_v8_original / "checkpoint_final.pt"
)

if not ruta_checkpoint_v8.exists():
    ruta_checkpoint_v8 = (
        directorio_v8_original / "ultimo_checkpoint.pt"
    )

if not ruta_checkpoint_v8.exists():
    raise FileNotFoundError(
        "No se encontró el checkpoint final de V8."
    )

globals().pop("mejor_dqn_v8", None)
globals().pop("checkpoint_v8", None)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

print("Checkpoint inicial:", ruta_checkpoint_v8)
print("Entrenamiento desde 1,000,000 hasta 2,000,000 pasos")

resultado_v8_extendido = entrenar_dqn(
    config=config_v8_extendido,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
    ruta_checkpoint_inicial=ruta_checkpoint_v8,
)

print("\nENTRENAMIENTO EXTENDIDO DE V8 FINALIZADO")
print(
    "Mejor promedio durante la extensión: "
    f"{resultado_v8_extendido['mejor_promedio_evaluacion']:.2f}"
)

Checkpoint inicial: ../models/v8_dqn_per_a06_5step/checkpoint_final.pt
Entrenamiento desde 1,000,000 hasta 2,000,000 pasos
Reanudando entrenamiento desde el paso 1,000,000 (checkpoint: ../models/v8_dqn_per_a06_5step/checkpoint_final.pt)
Replay buffer: priorizado | alpha=0.6 | beta inicial=0.4
Retorno utilizado: 5-step
Paso 1,010,000/2,000,000 | episodio=26 | epsilon=0.100 | loss=0.0714 | Q=4.895
Paso 1,011,000/2,000,000 | episodio=28 | epsilon=0.100 | loss=0.0274 | Q=4.426
Paso 1,012,000/2,000,000 | episodio=32 | epsilon=0.100 | loss=0.0294 | Q=5.167
Paso 1,013,000/2,000,000 | episodio=34 | epsilon=0.100 | loss=0.0430 | Q=5.041
Paso 1,014,000/2,000,000 | episodio=37 | epsilon=0.100 | loss=0.0075 | Q=4.991
Paso 1,015,000/2,000,000 | episodio=39 | epsilon=0.100 | loss=0.0172 | Q=4.800
Paso 1,016,000/2,000,000 | episodio=41 | epsilon=0.100 | loss=0.0097 | Q=5.049
Paso 1,017,000/2,000,000 | episodio=45 | epsilon=0.100 | loss=0.0320 | Q=4.523
Paso 1,018,000/2,000,000 | episodio=47 | epsilon

In [18]:
from pathlib import Path
import pandas as pd
from IPython.display import display


RUTA_LOG_V8_EXTENDIDO = Path(
    "../logs/entrenamientos/"
    "v8_dqn_per_a06_5step_extendido/"
    "evaluaciones.csv"
)

RUTA_MEJOR_MODELO_V8_EXTENDIDO = Path(
    "../models/"
    "v8_dqn_per_a06_5step_extendido/"
    "mejor_modelo.pt"
)

df_evaluaciones_v8_extendido = pd.read_csv(
    RUTA_LOG_V8_EXTENDIDO
)

display(df_evaluaciones_v8_extendido)

mejor_fila_v8_extendido = (
    df_evaluaciones_v8_extendido.loc[
        df_evaluaciones_v8_extendido["promedio"].idxmax()
    ]
)

promedio_ultimas_tres_v8_extendido = (
    df_evaluaciones_v8_extendido["promedio"]
    .tail(3)
    .mean()
)

print("\nMEJOR EVALUACIÓN DE V8 EXTENDIDO")
print(
    f"Paso: "
    f"{int(mejor_fila_v8_extendido['paso_global']):,}"
)
print(
    f"Promedio: "
    f"{mejor_fila_v8_extendido['promedio']:.2f}"
)
print(
    f"Mediana: "
    f"{mejor_fila_v8_extendido['mediana']:.2f}"
)
print(
    f"Desviación: "
    f"{mejor_fila_v8_extendido['desviacion']:.2f}"
)
print(
    f"Mínimo: "
    f"{mejor_fila_v8_extendido['minimo']:.2f}"
)
print(
    f"Máximo: "
    f"{mejor_fila_v8_extendido['maximo']:.2f}"
)

print(
    "\nPromedio de las últimas tres evaluaciones: "
    f"{promedio_ultimas_tres_v8_extendido:.2f}"
)

print(
    "\nCheckpoint disponible:",
    (
        "OK"
        if RUTA_MEJOR_MODELO_V8_EXTENDIDO.exists()
        else "NO ENCONTRADO"
    ),
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,1050000,467.0,450.0,98.315818,345.0,585.0
1,1100000,394.0,425.0,80.461171,280.0,490.0
2,1150000,421.0,410.0,159.417690,215.0,695.0
3,1200000,528.0,515.0,50.059964,465.0,600.0
4,1250000,523.0,570.0,149.251466,320.0,740.0
5,1300000,423.0,380.0,94.424573,320.0,570.0
6,1350000,368.0,365.0,124.120909,165.0,520.0
7,1400000,458.0,505.0,133.364163,250.0,600.0
8,1450000,264.0,275.0,50.833060,175.0,330.0
9,1500000,266.0,265.0,31.843367,220.0,310.0



MEJOR EVALUACIÓN DE V8 EXTENDIDO
Paso: 1,650,000
Promedio: 576.00
Mediana: 660.00
Desviación: 234.06
Mínimo: 130.00
Máximo: 770.00

Promedio de las últimas tres evaluaciones: 466.33

Checkpoint disponible: OK


In [19]:
import gc
from pathlib import Path

import torch

from models import DQN


RUTA_MEJOR_V8_EXTENDIDO = Path(
    "../models/"
    "v8_dqn_per_a06_5step_extendido/"
    "mejor_modelo.pt"
)

checkpoint_v8_extendido = torch.load(
    RUTA_MEJOR_V8_EXTENDIDO,
    map_location=device,
    weights_only=True,
)

mejor_v8_extendido = DQN(
    n_acciones=6
).to(device)

mejor_v8_extendido.load_state_dict(
    checkpoint_v8_extendido["modelo_online_state_dict"]
)

mejor_v8_extendido.eval()

print(
    "Checkpoint cargado desde el paso:",
    f"{checkpoint_v8_extendido['paso']:,}",
)

Checkpoint cargado desde el paso: 1,650,000


In [20]:
N_EPISODIOS_EVALUACION = 30
SEMILLA_BASE_EVALUACION = 42

print(
    "\nEvaluando V8 extendido durante "
    f"{N_EPISODIOS_EVALUACION} episodios...\n"
)

resultados_v8_extendido, resumen_v8_extendido = evaluar_modelo(
    modelo=mejor_v8_extendido,
    config=config_v8_extendido,
    device=device,
    n_episodios=N_EPISODIOS_EVALUACION,
    seed_base=SEMILLA_BASE_EVALUACION,
)

df_v8_extendido = pd.DataFrame(
    resultados_v8_extendido
)

df_v8_extendido["agente"] = (
    "DQN + PER a=0.6 + 5-step extendido"
)

df_v8_extendido = df_v8_extendido[
    [
        "agente",
        "episodio",
        "seed",
        "recompensa_total",
        "pasos",
        "terminated",
        "truncated",
    ]
]

print("RESUMEN DE 30 EPISODIOS")

for metrica, valor in resumen_v8_extendido.items():
    print(f"{metrica}: {valor:.2f}")

display(df_v8_extendido.head(10))


Evaluando V8 extendido durante 30 episodios...

RESUMEN DE 30 EPISODIOS
promedio: 455.33
mediana: 472.50
desviacion: 125.69
minimo: 190.00
maximo: 695.00


,agente,episodio,seed,recompensa_total,pasos,terminated,truncated
0,DQN + PER a=0.6 + 5-step extendido,0,42,190.0,483,True,False
1,DQN + PER a=0.6 + 5-step extendido,1,43,470.0,878,True,False
2,DQN + PER a=0.6 + 5-step extendido,2,44,525.0,1062,True,False
3,DQN + PER a=0.6 + 5-step extendido,3,45,335.0,732,True,False
4,DQN + PER a=0.6 + 5-step extendido,4,46,310.0,584,True,False
5,DQN + PER a=0.6 + 5-step extendido,5,47,350.0,643,True,False
6,DQN + PER a=0.6 + 5-step extendido,6,48,590.0,1035,True,False
7,DQN + PER a=0.6 + 5-step extendido,7,49,575.0,1029,True,False
8,DQN + PER a=0.6 + 5-step extendido,8,50,635.0,896,True,False
9,DQN + PER a=0.6 + 5-step extendido,9,51,475.0,683,True,False


In [21]:
from pathlib import Path

print("CARPETAS RELACIONADAS CON 3-STEP")

for ruta in sorted(Path("../models").glob("*3*step*")):
    print(ruta)

CARPETAS RELACIONADAS CON 3-STEP
../models/smoke_test_dqn_per_3step
../models/v6_dqn_per_3step
../models/v7_dqn_per_a04_3step
../models/v7_dqn_per_a04_3step_extendido


## Extención de la primera variación de hiperparámetros

In [22]:
import gc
from dataclasses import replace
from pathlib import Path

import torch

from train import entrenar_dqn
from models import DQN


RUTA_CHECKPOINT_V6 = Path(
    "../models/v6_dqn_per_3step/mejor_modelo.pt"
)

if not RUTA_CHECKPOINT_V6.exists():
    raise FileNotFoundError(
        f"No se encontró el checkpoint: {RUTA_CHECKPOINT_V6}"
    )

config_v6_extendido = replace(
    config_per_alpha06_5step,
    nombre_experimento="v6_dqn_per_3step_extendido",
    total_pasos=2_000_000,
    n_step=3,
)

objetos_temporales = [
    "mejor_v8_extendido",
    "checkpoint_v8_extendido",
    "resultado_v8_extendido",
]

for nombre in objetos_temporales:
    globals().pop(nombre, None)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

print("Checkpoint inicial:", RUTA_CHECKPOINT_V6)
print("Configuración: PER alpha=0.6 | retorno 3-step")
print("Entrenamiento desde 700,000 hasta 2,000,000 pasos")

resultado_v6_extendido = entrenar_dqn(
    config=config_v6_extendido,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
    ruta_checkpoint_inicial=RUTA_CHECKPOINT_V6,
)

print("\nENTRENAMIENTO EXTENDIDO DE V6 FINALIZADO")
print(
    "Mejor promedio durante la extensión: "
    f"{resultado_v6_extendido['mejor_promedio_evaluacion']:.2f}"
)

Checkpoint inicial: ../models/v6_dqn_per_3step/mejor_modelo.pt
Configuración: PER alpha=0.6 | retorno 3-step
Entrenamiento desde 700,000 hasta 2,000,000 pasos
Reanudando entrenamiento desde el paso 700,000 (checkpoint: ../models/v6_dqn_per_3step/mejor_modelo.pt)
Replay buffer: priorizado | alpha=0.6 | beta inicial=0.4
Retorno utilizado: 3-step
Paso 710,000/2,000,000 | episodio=27 | epsilon=0.100 | loss=0.1023 | Q=4.689
Paso 711,000/2,000,000 | episodio=29 | epsilon=0.100 | loss=0.0189 | Q=4.360
Paso 712,000/2,000,000 | episodio=33 | epsilon=0.100 | loss=0.0084 | Q=4.269
Paso 713,000/2,000,000 | episodio=36 | epsilon=0.100 | loss=0.0189 | Q=4.730
Paso 714,000/2,000,000 | episodio=40 | epsilon=0.100 | loss=0.0112 | Q=4.355
Paso 715,000/2,000,000 | episodio=41 | epsilon=0.100 | loss=0.0072 | Q=4.891
Paso 716,000/2,000,000 | episodio=43 | epsilon=0.100 | loss=0.0054 | Q=4.495
Paso 717,000/2,000,000 | episodio=46 | epsilon=0.100 | loss=0.0151 | Q=4.454
Paso 718,000/2,000,000 | episodio=49 |

In [23]:
from pathlib import Path
import pandas as pd
from IPython.display import display


RUTA_LOG_V6_EXTENDIDO = Path(
    "../logs/entrenamientos/"
    "v6_dqn_per_3step_extendido/"
    "evaluaciones.csv"
)

RUTA_MEJOR_MODELO_V6_EXTENDIDO = Path(
    "../models/"
    "v6_dqn_per_3step_extendido/"
    "mejor_modelo.pt"
)

df_evaluaciones_v6_extendido = pd.read_csv(
    RUTA_LOG_V6_EXTENDIDO
)

display(df_evaluaciones_v6_extendido)

mejor_fila_v6_extendido = (
    df_evaluaciones_v6_extendido.loc[
        df_evaluaciones_v6_extendido["promedio"].idxmax()
    ]
)

promedio_ultimas_tres_v6_extendido = (
    df_evaluaciones_v6_extendido["promedio"]
    .tail(3)
    .mean()
)

print("\nMEJOR EVALUACIÓN DE V6 EXTENDIDO")
print(
    f"Paso: "
    f"{int(mejor_fila_v6_extendido['paso_global']):,}"
)
print(
    f"Promedio: "
    f"{mejor_fila_v6_extendido['promedio']:.2f}"
)
print(
    f"Mediana: "
    f"{mejor_fila_v6_extendido['mediana']:.2f}"
)
print(
    f"Desviación: "
    f"{mejor_fila_v6_extendido['desviacion']:.2f}"
)
print(
    f"Mínimo: "
    f"{mejor_fila_v6_extendido['minimo']:.2f}"
)
print(
    f"Máximo: "
    f"{mejor_fila_v6_extendido['maximo']:.2f}"
)

print(
    "\nPromedio de las últimas tres evaluaciones: "
    f"{promedio_ultimas_tres_v6_extendido:.2f}"
)

print(
    "\nCheckpoint disponible:",
    (
        "OK"
        if RUTA_MEJOR_MODELO_V6_EXTENDIDO.exists()
        else "NO ENCONTRADO"
    ),
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,750000,403.0,425.0,55.641711,295.0,455.0
1,800000,366.0,380.0,59.782941,270.0,435.0
2,850000,433.0,395.0,111.651243,315.0,645.0
3,900000,414.0,410.0,58.000000,335.0,515.0
4,950000,501.0,495.0,83.390647,420.0,650.0
5,1000000,454.0,440.0,47.895720,405.0,545.0
6,1050000,418.0,475.0,127.734099,185.0,555.0
7,1100000,408.0,390.0,112.986725,250.0,545.0
8,1150000,386.0,370.0,96.405394,265.0,545.0
9,1200000,434.0,515.0,129.938447,210.0,555.0



MEJOR EVALUACIÓN DE V6 EXTENDIDO
Paso: 2,000,000
Promedio: 514.00
Mediana: 480.00
Desviación: 80.89
Mínimo: 440.00
Máximo: 650.00

Promedio de las últimas tres evaluaciones: 470.00

Checkpoint disponible: OK


In [24]:
import gc
from pathlib import Path

import torch

from models import DQN


RUTA_MEJOR_V6_EXTENDIDO = Path(
    "../models/"
    "v6_dqn_per_3step_extendido/"
    "mejor_modelo.pt"
)

checkpoint_v6_extendido = torch.load(
    RUTA_MEJOR_V6_EXTENDIDO,
    map_location=device,
    weights_only=True,
)

mejor_v6_extendido = DQN(
    n_acciones=6
).to(device)

mejor_v6_extendido.load_state_dict(
    checkpoint_v6_extendido["modelo_online_state_dict"]
)

mejor_v6_extendido.eval()

print(
    "Checkpoint cargado desde el paso:",
    f"{checkpoint_v6_extendido['paso']:,}",
)

Checkpoint cargado desde el paso: 2,000,000


In [25]:
N_EPISODIOS_EVALUACION = 30
SEMILLA_BASE_EVALUACION = 42

print(
    "\nEvaluando V6 extendido durante "
    f"{N_EPISODIOS_EVALUACION} episodios...\n"
)

resultados_v6_extendido, resumen_v6_extendido = evaluar_modelo(
    modelo=mejor_v6_extendido,
    config=config_v6_extendido,
    device=device,
    n_episodios=N_EPISODIOS_EVALUACION,
    seed_base=SEMILLA_BASE_EVALUACION,
)

df_v6_extendido = pd.DataFrame(
    resultados_v6_extendido
)

df_v6_extendido["agente"] = (
    "DQN + PER a=0.6 + 3-step extendido"
)

df_v6_extendido = df_v6_extendido[
    [
        "agente",
        "episodio",
        "seed",
        "recompensa_total",
        "pasos",
        "terminated",
        "truncated",
    ]
]

print("RESUMEN DE 30 EPISODIOS")

for metrica, valor in resumen_v6_extendido.items():
    print(f"{metrica}: {valor:.2f}")

display(df_v6_extendido.head(10))


Evaluando V6 extendido durante 30 episodios...

RESUMEN DE 30 EPISODIOS
promedio: 509.17
mediana: 497.50
desviacion: 113.39
minimo: 360.00
maximo: 910.00


,agente,episodio,seed,recompensa_total,pasos,terminated,truncated
0,DQN + PER a=0.6 + 3-step extendido,0,42,470.0,708,True,False
1,DQN + PER a=0.6 + 3-step extendido,1,43,360.0,598,True,False
2,DQN + PER a=0.6 + 3-step extendido,2,44,555.0,981,True,False
3,DQN + PER a=0.6 + 3-step extendido,3,45,615.0,803,True,False
4,DQN + PER a=0.6 + 3-step extendido,4,46,555.0,1134,True,False
5,DQN + PER a=0.6 + 3-step extendido,5,47,580.0,631,True,False
6,DQN + PER a=0.6 + 3-step extendido,6,48,385.0,732,True,False
7,DQN + PER a=0.6 + 3-step extendido,7,49,605.0,718,True,False
8,DQN + PER a=0.6 + 3-step extendido,8,50,460.0,613,True,False
9,DQN + PER a=0.6 + 3-step extendido,9,51,400.0,785,True,False


In [26]:
import gc
from dataclasses import replace
from pathlib import Path

import torch

from train import entrenar_dqn
from models import DQN


RUTA_CHECKPOINT_V6_2M = Path(
    "../models/"
    "v6_dqn_per_3step_extendido/"
    "mejor_modelo.pt"
)

if not RUTA_CHECKPOINT_V6_2M.exists():
    raise FileNotFoundError(
        f"No se encontró: {RUTA_CHECKPOINT_V6_2M}"
    )

config_v6_3m = replace(
    config_v6_extendido,
    nombre_experimento="v6_dqn_per_3step_3m",
    total_pasos=3_000_000,
)

objetos_temporales = [
    "mejor_v6_extendido",
    "checkpoint_v6_extendido",
    "resultado_v6_extendido",
]

for nombre in objetos_temporales:
    globals().pop(nombre, None)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

print("Checkpoint inicial:", RUTA_CHECKPOINT_V6_2M)
print("Continuación desde 2,000,000 hasta 3,000,000 pasos")
print("Modelo actual de 2M preservado en su carpeta original")

resultado_v6_3m = entrenar_dqn(
    config=config_v6_3m,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
    ruta_checkpoint_inicial=RUTA_CHECKPOINT_V6_2M,
)

print("\nENTRENAMIENTO HASTA 3M FINALIZADO")
print(
    "Mejor promedio durante esta extensión: "
    f"{resultado_v6_3m['mejor_promedio_evaluacion']:.2f}"
)

Checkpoint inicial: ../models/v6_dqn_per_3step_extendido/mejor_modelo.pt
Continuación desde 2,000,000 hasta 3,000,000 pasos
Modelo actual de 2M preservado en su carpeta original
Reanudando entrenamiento desde el paso 2,000,000 (checkpoint: ../models/v6_dqn_per_3step_extendido/mejor_modelo.pt)
Replay buffer: priorizado | alpha=0.6 | beta inicial=0.4
Retorno utilizado: 3-step
Paso 2,010,000/3,000,000 | episodio=22 | epsilon=0.100 | loss=0.0383 | Q=5.368
Paso 2,011,000/3,000,000 | episodio=25 | epsilon=0.100 | loss=0.0444 | Q=5.422
Paso 2,012,000/3,000,000 | episodio=27 | epsilon=0.100 | loss=0.0096 | Q=5.946
Paso 2,013,000/3,000,000 | episodio=30 | epsilon=0.100 | loss=0.0182 | Q=5.690
Paso 2,014,000/3,000,000 | episodio=31 | epsilon=0.100 | loss=0.0111 | Q=5.254
Paso 2,015,000/3,000,000 | episodio=34 | epsilon=0.100 | loss=0.0024 | Q=5.801
Paso 2,016,000/3,000,000 | episodio=36 | epsilon=0.100 | loss=0.0051 | Q=5.841
Paso 2,017,000/3,000,000 | episodio=39 | epsilon=0.100 | loss=0.0192 |

In [27]:
from pathlib import Path
import pandas as pd
from IPython.display import display


RUTA_LOG_V6_3M = Path(
    "../logs/entrenamientos/"
    "v6_dqn_per_3step_3m/"
    "evaluaciones.csv"
)

RUTA_MEJOR_MODELO_V6_3M = Path(
    "../models/"
    "v6_dqn_per_3step_3m/"
    "mejor_modelo.pt"
)

df_evaluaciones_v6_3m = pd.read_csv(
    RUTA_LOG_V6_3M
)

display(df_evaluaciones_v6_3m)

mejor_fila_v6_3m = df_evaluaciones_v6_3m.loc[
    df_evaluaciones_v6_3m["promedio"].idxmax()
]

promedio_ultimas_tres_v6_3m = (
    df_evaluaciones_v6_3m["promedio"]
    .tail(3)
    .mean()
)

print("\nMEJOR EVALUACIÓN ENTRE 2M Y 3M")
print(
    f"Paso: {int(mejor_fila_v6_3m['paso_global']):,}"
)
print(
    f"Promedio: {mejor_fila_v6_3m['promedio']:.2f}"
)
print(
    f"Mediana: {mejor_fila_v6_3m['mediana']:.2f}"
)
print(
    f"Desviación: {mejor_fila_v6_3m['desviacion']:.2f}"
)
print(
    f"Mínimo: {mejor_fila_v6_3m['minimo']:.2f}"
)
print(
    f"Máximo: {mejor_fila_v6_3m['maximo']:.2f}"
)

print(
    "\nPromedio de las últimas tres evaluaciones: "
    f"{promedio_ultimas_tres_v6_3m:.2f}"
)

print(
    "\nCheckpoint disponible:",
    (
        "OK"
        if RUTA_MEJOR_MODELO_V6_3M.exists()
        else "NO ENCONTRADO"
    ),
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,2050000,571.0,650.0,150.512458,285.0,690.0
1,2100000,424.0,410.0,73.579889,330.0,525.0
2,2150000,407.0,380.0,81.523003,305.0,505.0
3,2200000,470.0,450.0,63.560994,400.0,550.0
4,2250000,426.0,400.0,81.141851,335.0,575.0
5,2300000,482.0,475.0,137.971011,310.0,715.0
6,2350000,524.0,470.0,82.668011,435.0,645.0
7,2400000,362.0,330.0,114.830310,225.0,570.0
8,2450000,522.0,490.0,113.692568,350.0,685.0
9,2500000,625.0,600.0,103.730420,520.0,800.0



MEJOR EVALUACIÓN ENTRE 2M Y 3M
Paso: 2,700,000
Promedio: 635.00
Mediana: 740.00
Desviación: 146.32
Mínimo: 375.00
Máximo: 745.00

Promedio de las últimas tres evaluaciones: 552.33

Checkpoint disponible: OK


In [28]:
from pathlib import Path

import torch

from models import DQN


RUTA_MEJOR_V6_3M = Path(
    "../models/"
    "v6_dqn_per_3step_3m/"
    "mejor_modelo.pt"
)

checkpoint_v6_3m = torch.load(
    RUTA_MEJOR_V6_3M,
    map_location=device,
    weights_only=True,
)

mejor_v6_3m = DQN(
    n_acciones=6
).to(device)

mejor_v6_3m.load_state_dict(
    checkpoint_v6_3m["modelo_online_state_dict"]
)

mejor_v6_3m.eval()

print(
    "Checkpoint cargado desde el paso:",
    f"{checkpoint_v6_3m['paso']:,}",
)

Checkpoint cargado desde el paso: 2,700,000


In [29]:
N_EPISODIOS_EVALUACION = 30
SEMILLA_BASE_EVALUACION = 42

print(
    "\nEvaluando V6 entrenado hasta 3M "
    "(checkpoint de 2.7M) durante "
    f"{N_EPISODIOS_EVALUACION} episodios...\n"
)

resultados_v6_3m, resumen_v6_3m = evaluar_modelo(
    modelo=mejor_v6_3m,
    config=config_v6_3m,
    device=device,
    n_episodios=N_EPISODIOS_EVALUACION,
    seed_base=SEMILLA_BASE_EVALUACION,
)

df_v6_3m = pd.DataFrame(
    resultados_v6_3m
)

df_v6_3m["agente"] = (
    "DQN + PER a=0.6 + 3-step (2.7M)"
)

df_v6_3m = df_v6_3m[
    [
        "agente",
        "episodio",
        "seed",
        "recompensa_total",
        "pasos",
        "terminated",
        "truncated",
    ]
]

print("RESUMEN DE 30 EPISODIOS")

for metrica, valor in resumen_v6_3m.items():
    print(f"{metrica}: {valor:.2f}")

display(df_v6_3m.head(10))


Evaluando V6 entrenado hasta 3M (checkpoint de 2.7M) durante 30 episodios...

RESUMEN DE 30 EPISODIOS
promedio: 510.67
mediana: 480.00
desviacion: 137.43
minimo: 325.00
maximo: 920.00


,agente,episodio,seed,recompensa_total,pasos,terminated,truncated
0,DQN + PER a=0.6 + 3-step (2.7M),0,42,330.0,615,True,False
1,DQN + PER a=0.6 + 3-step (2.7M),1,43,410.0,711,True,False
2,DQN + PER a=0.6 + 3-step (2.7M),2,44,600.0,909,True,False
3,DQN + PER a=0.6 + 3-step (2.7M),3,45,570.0,1003,True,False
4,DQN + PER a=0.6 + 3-step (2.7M),4,46,735.0,768,True,False
5,DQN + PER a=0.6 + 3-step (2.7M),5,47,530.0,866,True,False
6,DQN + PER a=0.6 + 3-step (2.7M),6,48,920.0,1079,True,False
7,DQN + PER a=0.6 + 3-step (2.7M),7,49,605.0,819,True,False
8,DQN + PER a=0.6 + 3-step (2.7M),8,50,470.0,831,True,False
9,DQN + PER a=0.6 + 3-step (2.7M),9,51,325.0,618,True,False


In [30]:
import numpy as np
import pandas as pd
from IPython.display import display


comparacion_checkpoints = (
    df_v6_extendido[
        ["seed", "recompensa_total"]
    ]
    .rename(
        columns={
            "recompensa_total": "V6 2.0M"
        }
    )
    .merge(
        df_v6_3m[
            ["seed", "recompensa_total"]
        ].rename(
            columns={
                "recompensa_total": "V6 2.7M"
            }
        ),
        on="seed",
        how="inner",
        validate="one_to_one",
    )
)

diferencia = (
    comparacion_checkpoints["V6 2.7M"]
    - comparacion_checkpoints["V6 2.0M"]
)

print("COMPARACIÓN PAREADA")
print(
    "Victorias del checkpoint 2.0M:",
    int((diferencia < 0).sum()),
)
print(
    "Victorias del checkpoint 2.7M:",
    int((diferencia > 0).sum()),
)
print(
    "Empates:",
    int((diferencia == 0).sum()),
)
print(
    "Diferencia promedio a favor de 2.7M:",
    f"{diferencia.mean():.2f} puntos",
)

df_bloques_checkpoints = pd.concat(
    [
        df_v6_extendido.assign(
            agente="V6 checkpoint 2.0M"
        ),
        df_v6_3m.assign(
            agente="V6 checkpoint 2.7M"
        ),
    ],
    ignore_index=True,
)

df_bloques_checkpoints["bloque_de_5"] = (
    df_bloques_checkpoints
    .groupby("agente")
    .cumcount()
    .floordiv(5)
    .add(1)
)

mejores_por_bloque = (
    df_bloques_checkpoints
    .groupby(
        ["agente", "bloque_de_5"],
        as_index=False,
    )
    .agg(
        mejor_recompensa=(
            "recompensa_total",
            "max",
        ),
        promedio_bloque=(
            "recompensa_total",
            "mean",
        ),
    )
)

resumen_mejor_de_5 = (
    mejores_por_bloque
    .groupby("agente", as_index=False)
    .agg(
        mejor_de_5_promedio=(
            "mejor_recompensa",
            "mean",
        ),
        mejor_de_5_mediana=(
            "mejor_recompensa",
            "median",
        ),
        menor_mejor_de_5=(
            "mejor_recompensa",
            "min",
        ),
        mayor_mejor_de_5=(
            "mejor_recompensa",
            "max",
        ),
    )
)

print("\nRESULTADOS POR BLOQUE")
display(mejores_por_bloque)

print("\nRESUMEN DEL MEJOR DE CINCO")
display(resumen_mejor_de_5)

COMPARACIÓN PAREADA
Victorias del checkpoint 2.0M: 16
Victorias del checkpoint 2.7M: 13
Empates: 1
Diferencia promedio a favor de 2.7M: 1.50 puntos

RESULTADOS POR BLOQUE


,agente,bloque_de_5,mejor_recompensa,promedio_bloque
0,V6 checkpoint 2.0M,1,615.0,511.0
1,V6 checkpoint 2.0M,2,605.0,486.0
2,V6 checkpoint 2.0M,3,610.0,511.0
3,V6 checkpoint 2.0M,4,700.0,516.0
4,V6 checkpoint 2.0M,5,570.0,492.0
5,V6 checkpoint 2.0M,6,910.0,539.0
6,V6 checkpoint 2.7M,1,735.0,529.0
7,V6 checkpoint 2.7M,2,920.0,570.0
8,V6 checkpoint 2.7M,3,690.0,539.0
9,V6 checkpoint 2.7M,4,715.0,517.0



RESUMEN DEL MEJOR DE CINCO


,agente,mejor_de_5_promedio,mejor_de_5_mediana,menor_mejor_de_5,mayor_mejor_de_5
0,V6 checkpoint 2.0M,668.333333,612.5,570.0,910.0
1,V6 checkpoint 2.7M,688.333333,702.5,500.0,920.0


### Selección del modelo final

El modelo seleccionado fue **DQN con Prioritized Experience Replay
($\alpha=0.6$) y retornos de 3 pasos**, utilizando el checkpoint obtenido
en el paso **2,700,000**.

En una evaluación greedy de 30 episodios alcanzó un promedio de
**510.67 puntos**, una mediana de **480 puntos** y un máximo de
**920 puntos**.

Su rendimiento promedio fue prácticamente equivalente al checkpoint de
2,000,000 pasos. Sin embargo, obtuvo mejores resultados bajo el criterio
de mejor episodio en bloques de cinco: su mejor-de-cinco promedio fue
**688.33**, frente a **668.33** del checkpoint de 2,000,000 pasos.

Aunque el checkpoint de 2,000,000 ganó más comparaciones pareadas y
presentó menor variabilidad, se eligió el de 2,700,000 porque el criterio
de evaluación de la competencia favorece el mejor desempeño obtenido
durante cinco episodios. El checkpoint de 2,000,000 se conservó como
modelo de respaldo por su mayor estabilidad.

In [31]:
from pathlib import Path

DIRECTORIO_RESULTADOS = Path("../logs/resultados_finales")
DIRECTORIO_RESULTADOS.mkdir(
    parents=True,
    exist_ok=True,
)

RUTA_EVALUACION_FINAL = (
    DIRECTORIO_RESULTADOS
    / "evaluacion_modelo_final_2_7m.csv"
)

RUTA_COMPARACION_CHECKPOINTS = (
    DIRECTORIO_RESULTADOS
    / "comparacion_checkpoints_2m_vs_2_7m.csv"
)

RUTA_MEJOR_DE_CINCO = (
    DIRECTORIO_RESULTADOS
    / "mejor_de_cinco_2m_vs_2_7m.csv"
)

df_v6_3m.to_csv(
    RUTA_EVALUACION_FINAL,
    index=False,
)

comparacion_checkpoints.to_csv(
    RUTA_COMPARACION_CHECKPOINTS,
    index=False,
)

resumen_mejor_de_5.to_csv(
    RUTA_MEJOR_DE_CINCO,
    index=False,
)

print("ARCHIVOS GUARDADOS")

for ruta in [
    RUTA_EVALUACION_FINAL,
    RUTA_COMPARACION_CHECKPOINTS,
    RUTA_MEJOR_DE_CINCO,
]:
    print(f"{ruta}: {'OK' if ruta.exists() else 'ERROR'}")

ARCHIVOS GUARDADOS
../logs/resultados_finales/evaluacion_modelo_final_2_7m.csv: OK
../logs/resultados_finales/comparacion_checkpoints_2m_vs_2_7m.csv: OK
../logs/resultados_finales/mejor_de_cinco_2m_vs_2_7m.csv: OK


In [32]:
import numpy as np
import torch

from ale_utils import generar_video_agente


def agente_greedy_final(observacion, env):
    observacion_np = np.asarray(
        observacion,
        dtype=np.uint8,
    )

    observacion_tensor = torch.as_tensor(
        observacion_np,
        device=device,
    ).unsqueeze(0)

    with torch.no_grad():
        valores_q = mejor_v6_3m(
            observacion_tensor
        )

    accion = valores_q.argmax(
        dim=1
    ).item()

    return accion


print("Agente greedy final preparado.")

Agente greedy final preparado.


In [41]:
from pathlib import Path
import shutil
import pandas as pd


DIRECTORIO_VIDEOS = Path(
    "../videos/candidatos_modelo_final"
)

resultado_videos_finales = generar_video_agente(
    nombre_entorno="ALE/SpaceInvaders-v5",
    funcion_agente=agente_greedy_final,
    video_folder=str(DIRECTORIO_VIDEOS),
    name_prefix="dqn_per_3step_2_7m",
    n_episodios=5,
    max_steps=10_000,

    aplicar_preprocesamiento=True,
    frame_skip=4,
    screen_size=84,
    grayscale=True,
    stack_size=4,

    # Evaluación de la partida completa
    terminal_on_life_loss=False,
    clip_reward=False,
    noop_max=30,

    full_action_space=False,
)

df_videos_finales = pd.DataFrame(
    resultado_videos_finales["metricas"]
)

df_videos_finales["ruta_video"] = (
    resultado_videos_finales["videos"]
)

display(df_videos_finales)

indice_mejor_video = (
    df_videos_finales["recompensa_total"].idxmax()
)

mejor_video = Path(
    df_videos_finales.loc[
        indice_mejor_video,
        "ruta_video",
    ]
)

DIRECTORIO_FINAL = Path("../videos")
DIRECTORIO_FINAL.mkdir(
    parents=True,
    exist_ok=True,
)

RUTA_VIDEO_ENTREGABLE = (
    DIRECTORIO_FINAL
    / "space_invaders_modelo_final.mp4"
)

shutil.copy2(
    mejor_video,
    RUTA_VIDEO_ENTREGABLE,
)

print("\nVIDEO SELECCIONADO")
print(
    "Episodio:",
    int(
        df_videos_finales.loc[
            indice_mejor_video,
            "episodio",
        ]
    ),
)
print(
    "Puntuación:",
    df_videos_finales.loc[
        indice_mejor_video,
        "recompensa_total",
    ],
)
print(
    "Pasos:",
    int(
        df_videos_finales.loc[
            indice_mejor_video,
            "pasos",
        ]
    ),
)
print("Archivo:", RUTA_VIDEO_ENTREGABLE)
print(
    "Existe:",
    RUTA_VIDEO_ENTREGABLE.exists(),
)

/Users/Camila/Desktop/CAMILA UNIVERSIDAD/8SEMESTRE/DeepLearning/Proyecto2-DL/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:283: UserWarning: WARN: Overwriting existing videos at /Users/Camila/Desktop/CAMILA UNIVERSIDAD/8SEMESTRE/DeepLearning/Proyecto2-DL/videos/candidatos_modelo_final folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


,pasos,recompensa_total,terminated,truncated,seed,episodio,ruta_video
0,823,465.0,True,False,None,0,../videos/candidatos_modelo_final/dqn_per_3ste...
1,1009,600.0,True,False,None,1,../videos/candidatos_modelo_final/dqn_per_3ste...
2,690,350.0,True,False,None,2,../videos/candidatos_modelo_final/dqn_per_3ste...
3,683,285.0,True,False,None,3,../videos/candidatos_modelo_final/dqn_per_3ste...
4,999,775.0,True,False,None,4,../videos/candidatos_modelo_final/dqn_per_3ste...



VIDEO SELECCIONADO
Episodio: 4
Puntuación: 775.0
Pasos: 999
Archivo: ../videos/space_invaders_modelo_final.mp4
Existe: True


In [42]:
from pathlib import Path
import shutil
import pandas as pd
from IPython.display import display


# Carpeta independiente para evitar mezclar ambas tandas
DIRECTORIO_VIDEOS_TANDA_2 = Path(
    "../videos/candidatos_modelo_final_tanda_2"
)

resultado_videos_tanda_2 = generar_video_agente(
    nombre_entorno="ALE/SpaceInvaders-v5",
    funcion_agente=agente_greedy_final,
    video_folder=str(DIRECTORIO_VIDEOS_TANDA_2),
    name_prefix="dqn_per_3step_2_7m_tanda_2",
    n_episodios=5,
    max_steps=10_000,

    aplicar_preprocesamiento=True,
    frame_skip=4,
    screen_size=84,
    grayscale=True,
    stack_size=4,

    # Partida completa con puntuación real
    terminal_on_life_loss=False,
    clip_reward=False,
    noop_max=30,

    full_action_space=False,
)

df_videos_tanda_2 = pd.DataFrame(
    resultado_videos_tanda_2["metricas"]
)

df_videos_tanda_2["ruta_video"] = (
    resultado_videos_tanda_2["videos"]
)

display(df_videos_tanda_2)

# Seleccionar el episodio con mayor puntuación
indice_mejor_tanda_2 = (
    df_videos_tanda_2["recompensa_total"].idxmax()
)

mejor_fila_tanda_2 = df_videos_tanda_2.loc[
    indice_mejor_tanda_2
]

mejor_video_tanda_2 = Path(
    mejor_fila_tanda_2["ruta_video"]
)

DIRECTORIO_FINAL = Path("../videos")
DIRECTORIO_FINAL.mkdir(
    parents=True,
    exist_ok=True,
)

# Guardarlo por separado para no sobrescribir todavía
RUTA_VIDEO_TANDA_2 = (
    DIRECTORIO_FINAL
    / "space_invaders_modelo_final_tanda_2.mp4"
)

shutil.copy2(
    mejor_video_tanda_2,
    RUTA_VIDEO_TANDA_2,
)

puntuacion_tanda_1 = 775.0
puntuacion_tanda_2 = float(
    mejor_fila_tanda_2["recompensa_total"]
)

print("\nMEJOR VIDEO DE LA SEGUNDA TANDA")
print(
    "Episodio:",
    int(mejor_fila_tanda_2["episodio"]),
)
print(
    "Puntuación:",
    puntuacion_tanda_2,
)
print(
    "Pasos:",
    int(mejor_fila_tanda_2["pasos"]),
)
print(
    "Terminated:",
    bool(mejor_fila_tanda_2["terminated"]),
)
print(
    "Truncated:",
    bool(mejor_fila_tanda_2["truncated"]),
)
print(
    "Archivo:",
    RUTA_VIDEO_TANDA_2,
)
print(
    "Existe:",
    RUTA_VIDEO_TANDA_2.exists(),
)

print("\nCOMPARACIÓN CON LA PRIMERA TANDA")
print(
    f"Primera tanda: {puntuacion_tanda_1:.0f} puntos"
)
print(
    f"Segunda tanda: {puntuacion_tanda_2:.0f} puntos"
)

if puntuacion_tanda_2 > puntuacion_tanda_1:
    print(
        "La segunda tanda obtuvo un video mejor."
    )
else:
    print(
        "Se conserva el video de la primera tanda."
    )

/Users/Camila/Desktop/CAMILA UNIVERSIDAD/8SEMESTRE/DeepLearning/Proyecto2-DL/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:283: UserWarning: WARN: Overwriting existing videos at /Users/Camila/Desktop/CAMILA UNIVERSIDAD/8SEMESTRE/DeepLearning/Proyecto2-DL/videos/candidatos_modelo_final_tanda_2 folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


,pasos,recompensa_total,terminated,truncated,seed,episodio,ruta_video
0,623,330.0,True,False,None,0,../videos/candidatos_modelo_final_tanda_2/dqn_...
1,1138,800.0,True,False,None,1,../videos/candidatos_modelo_final_tanda_2/dqn_...
2,727,420.0,True,False,None,2,../videos/candidatos_modelo_final_tanda_2/dqn_...
3,626,535.0,True,False,None,3,../videos/candidatos_modelo_final_tanda_2/dqn_...
4,744,495.0,True,False,None,4,../videos/candidatos_modelo_final_tanda_2/dqn_...



MEJOR VIDEO DE LA SEGUNDA TANDA
Episodio: 1
Puntuación: 800.0
Pasos: 1138
Terminated: True
Truncated: False
Archivo: ../videos/space_invaders_modelo_final_tanda_2.mp4
Existe: True

COMPARACIÓN CON LA PRIMERA TANDA
Primera tanda: 775 puntos
Segunda tanda: 800 puntos
La segunda tanda obtuvo un video mejor.


In [43]:
from pathlib import Path
import shutil
from IPython.display import Video, display


RUTA_VIDEO_ANTERIOR = Path(
    "../videos/space_invaders_modelo_final.mp4"
)

RUTA_RESPALDO_775 = Path(
    "../videos/space_invaders_respaldo_775_puntos.mp4"
)

RUTA_VIDEO_800 = Path(
    "../videos/space_invaders_modelo_final_tanda_2.mp4"
)

RUTA_VIDEO_FINAL = Path(
    "../videos/space_invaders_modelo_final.mp4"
)

# Conservar el video anterior como respaldo
if (
    RUTA_VIDEO_ANTERIOR.exists()
    and not RUTA_RESPALDO_775.exists()
):
    shutil.copy2(
        RUTA_VIDEO_ANTERIOR,
        RUTA_RESPALDO_775,
    )

# Establecer el video de 800 puntos como entregable
shutil.copy2(
    RUTA_VIDEO_800,
    RUTA_VIDEO_FINAL,
)

tamano_mb = (
    RUTA_VIDEO_FINAL.stat().st_size
    / (1024 ** 2)
)

print("VIDEO FINAL ACTUALIZADO")
print("Puntuación: 800 puntos")
print("Pasos: 1,138")
print("Terminated: True")
print("Truncated: False")
print("Archivo:", RUTA_VIDEO_FINAL)
print(f"Tamaño: {tamano_mb:.2f} MB")
print("Respaldo de 775:", RUTA_RESPALDO_775.exists())

display(
    Video(
        str(RUTA_VIDEO_FINAL),
        embed=True,
    )
)

VIDEO FINAL ACTUALIZADO
Puntuación: 800 puntos
Pasos: 1,138
Terminated: True
Truncated: False
Archivo: ../videos/space_invaders_modelo_final.mp4
Tamaño: 0.48 MB
Respaldo de 775: True
